# 4. Document Q&A with RAG

Answer support questions using retrieved evidence from local policy documents.

[🔊 Open Interview & Audio Practice](https://htmlpreview.github.io/?https://github.com/blaire101/langchain-course-companion-26/blob/main/docs/04_document_qa.html)

> GitHub notebook previews do not execute custom JavaScript. Interactive pronunciation and interview notes are provided in the companion HTML page.


## 1. Import RAG components

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
MODEL = "openai:gpt-4o-mini"

## 2. Load, chunk, embed, and index documents

In [ ]:
documents = []
for path in Path("../data").glob("*.txt"):
    loaded = TextLoader(str(path), encoding="utf-8").load()
    for document in loaded:
        document.metadata["file_name"] = path.name
    documents.extend(loaded)

chunks = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
).split_documents(documents)

store = FAISS.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small"),
)

## 3. Retrieve evidence and generate an answer

In [ ]:
question = "Which plan includes audit logs?"
retrieved = store.similarity_search(question, k=3)

context = "\n\n".join(
    f"SOURCE: {doc.metadata['file_name']}\n{doc.page_content}"
    for doc in retrieved
)

model = init_chat_model(MODEL, temperature=0)
response = model.invoke(
    "Answer only from the context and cite the source file.\n\n"
    f"{context}\n\nQuestion: {question}"
)
print(response.content)

## Example Output

```text
The Business plan includes audit logs. Source: product_guide.txt
```

The exact wording may vary for model-generated responses, while the expected facts and structure should remain consistent.


## Relationship Diagram

![Document Q&A with RAG flow](../assets/04_document_qa_flow.png)

### Relationship Summary

- **1. Policy Files**
- **2. Load & Chunk**
- **3. Embeddings**
- **4. FAISS**
- **5. Top-k Retrieval**
- **6. gpt-4o-mini**
- **7. Answer + Source**

## Think Summary

### How do chunk size and top-k affect retrieval?

- Small chunks improve precision but may lose surrounding context.
- Large chunks preserve context but consume more tokens and may include irrelevant text.
- A larger top-k improves recall but can add noise and cost.
- Retrieval and generation should be evaluated separately.

### Key takeaway

Retrieval and generation should be evaluated separately.
